# Polars Benchmark

This notebook runs an representative end to end use case over data sourced from the [UK Land Registry House Price Data open data repository](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads).

This data is made available for us under an [Open Government Licence](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).

We will run two processes:

1. Load raw data, clean it up, add new features and finally write it as a mini dimensional model to lakehouse.
1. Query two of the tables in the dimensional model, join them and summarise the data.

In [15]:
import polars as pl
import time
import logging

In [16]:
logger = logging.getLogger(name="polars_benchmark_notebook")
logger.setLevel(logging.INFO)

In [17]:
from datetime import datetime

source_path = "../../data/fabric/Files/land_registry_data" # ABFSS path to location where raw data (multiple CSV files) is stored
storage_options = {}

# Add timestamp to paths to avoid overwrite issues with Parquet
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
target_path_prices = f"../../data/fabric/Tables/polars_benchmark/{run_timestamp}/prices.parquet"
target_path_locations = f"../../data/fabric/Tables/polars_benchmark/{run_timestamp}/locations.parquet"
target_path_dates = f"../../data/fabric/Tables/polars_benchmark/{run_timestamp}/dates.parquet"

In [18]:
start = time.perf_counter()

In [19]:
logging.info(f"Reading price paid data from location {source_path}...")

# Files area
price_paid_data = pl.scan_csv(
    source_path,  # AFBSS path to the CSV files in the Files area.
    has_header=False,
    null_values=[""],
    storage_options=storage_options,  # Provides Polars with the necessary credentials to read from Fabric.
    infer_schema=False,
    schema={
        "transaction_unique_identifier": pl.Utf8,
        "price": pl.Float64,
        "date_of_transfer": pl.Datetime,
        "postcode": pl.Utf8,
        "property_type": pl.Utf8,
        "old_new": pl.Utf8,
        "duration": pl.Utf8,
        "paon": pl.Utf8,
        "saon": pl.Utf8,
        "street": pl.Utf8,
        "locality": pl.Utf8,
        "town_city": pl.Utf8,
        "district": pl.Utf8,
        "county": pl.Utf8,
        "ppd_category_type": pl.Utf8,
        "record_status": pl.Utf8
    })

## Data Transformation

Now we can have a lazy frame in place, we can start to build up the transformations we want apply using Polars' composable expression API:

In [20]:
# Convert the property_type column from single letter codes to full descriptions
price_paid_data = (
    price_paid_data
    .with_columns(
        pl.when(pl.col("property_type") == "D")
        .then(pl.lit("Detached"))
        .when(pl.col("property_type") == "S")
        .then(pl.lit("Semi-Detached"))
        .when(pl.col("property_type") == "T")
        .then(pl.lit("Terraced"))
        .when(pl.col("property_type") == "F")
        .then(pl.lit("Flat/Maisonette"))
        .when(pl.col("property_type") == "O")
        .then(pl.lit("Other"))
        .otherwise(pl.col("property_type"))
        .alias("property_type")
    )
)

In [21]:
# Do the same of old_new
price_paid_data = (
    price_paid_data
    .with_columns(
        pl.when(pl.col("old_new") == "Y")
        .then(pl.lit("New"))
        .when(pl.col("old_new") == "N")
        .then(pl.lit("Old"))
        .otherwise(pl.col("old_new"))
        .alias("old_new")
    )
)

In [22]:
# Use regex to extract the postcode area (the first one or two letters)
price_paid_data = (
    price_paid_data
    .with_columns(
        pl.col("postcode")
        .str.extract(r"^([A-Z]{1,2})", 1)
        .alias("postcode_area")
    )
)

In [23]:
# Convert date_of_transfer from datetime to date
price_paid_data = (
    price_paid_data
    .with_columns(
        pl.col("date_of_transfer")
        .dt.date()
        .alias("date_of_transfer")
    )
)

### Create fact table

Select the core columns we want to use in the core fact table.

In [24]:
# Select relevant columns for downstream analysis
prices = price_paid_data.select([
    "price",
    "date_of_transfer",
    "postcode_area",
    "town_city",
    "property_type",
    "old_new",
])

### Create date dimension

Use min and max dates to build date dimension table.

At this stage we need to materialise the data.  But given we are operating over a single column, the operaiton will be optimised through **projection pushdown**.

In [25]:
min_date = price_paid_data.select(pl.col("date_of_transfer").min()).collect()[0,0]
max_date = price_paid_data.select(pl.col("date_of_transfer").max()).collect()[0,0]
min_date, max_date

(datetime.date(2023, 1, 1), datetime.date(2025, 11, 28))

In [26]:
dates = (
    pl.date_range(
        start=min_date,
        end=max_date,
        interval="1d",
        eager=True,
    ).
    to_frame(name="date")
    .with_columns([
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.strftime("%B").alias("month_name"),
        pl.col("date").dt.day().alias("day"),
        pl.col("date").dt.weekday().alias("weekday"),
        pl.col("date").dt.strftime("%A").alias("weekday_name"),
        pl.col("date").dt.ordinal_day().alias("day_of_year"),
    ])
)   

### Create location dimension

Assumption is there is a hierarchy in descreasing order of granularity:

- County
- District
- Town or City

In [27]:
locations = (
    price_paid_data
    .select(
        [
            "county",
            "district",
            "town_city",
        ]
    )
    .unique()
)

## Writing to Delta Tables

It is common practice to write out a Polars DataFrame to a Delta table in the Tables area of your Lakehouse.

There are various write modes which are avialble:

Overwrite entire table:

```python
df.write_delta(path, mode="overwrite")
```

Append to existing table:

```python
df.write_delta(path, mode="append")
```

Merge (upsert) - returns a TableMerger for chaining:

```python
(
    df.write_delta(
        path,
        mode="merge",
        delta_merge_options={
            "predicate": "source.id = target.id",
            "source_alias": "source",
            "target_alias": "target"
        }
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)
```

### Handling Timestamps

A common gotcha when writing Delta tables from Polars is timezone handling. Fabric's SQL endpoint expects timestamps with timezone information.

We can address this by adding timezone information, for example:

```python
df = (
    df
    .with_columns(
        [
            pl.col("datetime_of_order")
            .dt.replace_time_zone("UTC")
            .alias("datetime_of_order")
        ]
    )
)
```

### Write tables

In [28]:
import os
os.makedirs(os.path.dirname(target_path_prices), exist_ok=True)

logger.info(f"Writing prices data to Parquet: {target_path_prices}")
prices.collect().write_parquet(target_path_prices)

INFO:polars_benchmark_notebook:Writing prices data to Parquet: ../../data/fabric/Tables/polars_benchmark/20260120_182620/prices.parquet


In [29]:
logger.info(f"Writing locations data to Parquet: {target_path_locations}")
locations.collect().write_parquet(target_path_locations)

INFO:polars_benchmark_notebook:Writing locations data to Parquet: ../../data/fabric/Tables/polars_benchmark/20260120_182620/locations.parquet


In [30]:
logger.info(f"Writing dates data to Parquet: {target_path_dates}")
dates.write_parquet(target_path_dates)

INFO:polars_benchmark_notebook:Writing dates data to Parquet: ../../data/fabric/Tables/polars_benchmark/20260120_182620/dates.parquet


## Reading from DeltaLake and generate sumamry

When we are reading delta files, we can use the Lazy execution framework to maximise scale and performance.

Let's illustrate this by doing generating some analytics in this notebook using the data we have just written to the lakehouse in Delta format.

In [31]:
# Load prices from Parquet and filter them to exclude "Other" property types
logger.info(f"Reading prices data back from Parquet: {target_path_prices}")
prices = (
    pl.scan_parquet(target_path_prices)
    .filter(pl.col("property_type") != "Other")
)

INFO:polars_benchmark_notebook:Reading prices data back from Parquet: ../../data/fabric/Tables/polars_benchmark/20260120_182620/prices.parquet


In [32]:
# Load the date dimension, add a new month_tag column in the form YYYY_MM
logger.info(f"Reading dates data back from Parquet: {target_path_dates}")
dates = (
    pl.scan_parquet(target_path_dates)
    .with_columns(
        [
            pl.col("date").dt.strftime("%Y_%m").alias("month_tag")
        ]
    )
)

INFO:polars_benchmark_notebook:Reading dates data back from Parquet: ../../data/fabric/Tables/polars_benchmark/20260120_182620/dates.parquet


In [33]:
# Now join the two tables to get month_tag into the prices table
prices = (
    prices
    .join(
        dates.select(
            [
                "date",
                "month_tag"
            ]
        ),
        left_on="date_of_transfer",
        right_on="date",
        how="left"
    )
)

In [34]:
# Finally summarise the data up to monthly level by property type
monthly_summary = (
    prices
    .group_by(
        [
            "month_tag",
            "property_type"
        ]
    )
    .agg(
        [
            pl.len().alias("number_of_transactions"),
            pl.col("price").median().alias("median_price"),
            pl.col("price").min().alias("min_price"),
            pl.col("price").max().alias("max_price"),
        ]
    )
    .sort(
        [
            "month_tag",
            "property_type"
        ]
    )
)

In [35]:
monthly_summary = monthly_summary.collect()

In [36]:
monthly_summary.head(5)

month_tag,property_type,number_of_transactions,median_price,min_price,max_price
str,str,u32,f64,f64,f64
"""2023_01""","""Detached""",26620,430000.0,33000.0,4.475e7
"""2023_01""","""Flat/Maisonette""",25958,239000.0,14000.0,1.35e7
"""2023_01""","""Semi-Detached""",33278,260000.0,950.0,2.8e7
"""2023_01""","""Terraced""",35592,212500.0,12500.0,1.615e7
"""2023_02""","""Detached""",26208,420000.0,39999.0,1.75e7


In [37]:
elapsed = time.perf_counter() - start
logger.info(f"Notebook completed in {elapsed:.2f} seconds.")

INFO:polars_benchmark_notebook:Notebook completed in 1.87 seconds.


## Summary

Polars on Microsoft Fabric offers a compelling alternative to Spark for many data engineering workloads. The combination of Polars' performance, Fabric's native OneLake integration, and the cost efficiency of single-node compute creates a practical path for teams who want enterprise-grade data pipelines without the complexity of distributed systems.

Start small, measure your workloads, and scale to Spark only when you genuinely need distributed compute. For many teams, that day may never come.